# NB4 — Cards: faithfulness & provenance (Lens 4 / M4)

One Layer C **card** per anchor theme; a **provenance** trace (cited chunks
resolved / total); and **extract-vs-card** (the Stage-1 extractive summary
beside the LLM card, to show refinement / drift).

**Requires a real LLM** (Groq `llama-3.3-70b-versatile`). The first cell guards
against the mock and aborts loudly — NB4 must never emit fabricated card text.
One salient theme per anchor is sent to the LLM (one live call each — three
total for the three anchors: olive_oil, legume, fish), not the whole facet.
The `fish` anchor is primarily abstract-driven and demonstrates journal
abstracts powering a card end-to-end.

In [1]:
import os, sys
sys.path.insert(0, ".")
import cs_common as cs

OUT = cs.artifacts_dir("nb4_cards")
plt = cs.init_mpl()
# GROQ_API_KEY must come from the environment:  export GROQ_API_KEY=...

print("artifacts ->", OUT)
# NEVER hardcode the key in the notebook. Export it in the shell before running:
#   export GROQ_API_KEY=...   (then: python _run_nb.py NB4_cards.ipynb)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

artifacts -> /mnt/workspaces/wisefood/foodscholar-lib/notebooks/case_study/artifacts/nb4_cards
GROQ_API_KEY present: True


## 1. Real-LLM facade (guarded) + reload graph

In [2]:
# get_fs(with_llm=True) builds the Groq client and raises if GROQ_API_KEY is
# absent — NB4 fails loudly rather than fabricating with the mock.
fs = cs.get_fs(with_llm=True)
assert not cs.is_mock_llm(fs), "refusing to run NB4 on the mock LLM"
print("LLM:", fs.llm.model_id)
loaded = cs.load_graph(fs, themes=True, cards=False)
print("reloaded:", loaded)
anchors = cs.resolve_anchors(fs)
env = cs.capture_env(fs, extra={"reloaded": loaded, "llm_model": fs.llm.model_id})
cs.save_json(OUT / "env.json", env)

LLM: llama-3.3-70b-versatile


2026-06-24T12:26:34.217852Z [info     ] corpus.loaded                  config_hash=2bc7a9068494781c n=34359


reloaded: {'chunks': 34359, 'shelves': 1019, 'themes': 41, 'cards': 0}


## 2. Pick one salient theme per anchor + build its card

We reuse the Layer C builder internals (`run_stage1` → `run_stage2`) for just
the chosen theme, so only two live LLM calls are made. The salient theme is the
largest theme on the anchor shelf (carried by `theme_id` from NB2).

In [3]:
from foodscholar.layer_c.registry import build_summarizer
from foodscholar.layer_c.stage1 import run_stage1
from foodscholar.layer_c.stage2 import run_stage2
from foodscholar.layer_c.builder import _ThemeAdapter, persist_cards

cfg = fs.config.layer_c
summarizer = build_summarizer(cfg.stage1_method, cfg)

# Salient theme = the on-topic theme for each anchor, selected by CONTENT only
# (keyword overlap with cue terms + abstract fraction + size + a chunk-set hash
# tiebreak) — never by the reshuffleable "<anchor>-leiden-N" id. Off-topic
# themes (zero cue overlap) are excluded; if none overlap we RAISE rather than
# silently fall back to a generic high-count theme. Among on-topic themes we
# select by `PREFER` (abstract-rich for legume/fish, textbook for olive_oil) —
# but here all current picks happen to be abstract-rich; PREFER stays as a knob.
# All ties are broken deterministically, so the same theme is chosen every run.
import re, hashlib
_STOP = {"the", "and", "of", "do", "are", "is", "what", "how", "good", "for", "a", "to"}
CUES = {
    "olive_oil": {"oil", "olive", "fat", "fats", "fatty", "monounsaturated", "saturated"},
    "legume": {"fiber", "fibre", "beans", "legume", "legumes", "pulses", "bean", "lentil",
               "cholesterol", "glycemic", "diabetes", "dietary", "intake"},
    "fish": {"omega-3", "omega", "fatty", "fats", "acids", "dha", "epa", "pufa", "pufas",
             "fish oil", "oil", "cardiovascular"},
    "dietary_fibre": {"fibre", "fiber", "soluble", "insoluble", "glycemic", "glucose",
             "cholesterol", "blood", "sugar", "prebiotic", "fermentable", "viscous"},
}
PREFER = {"olive_oil": "abstract", "legume": "abstract", "fish": "abstract", "dietary_fibre": "abstract"}

def _cue_terms(anchor_key):
    q = cs.CONFIG["queries"].get(anchor_key, "")
    qt = {w for w in re.findall(r"[a-z0-9-]+", q.lower()) if w not in _STOP and len(w) > 2}
    return qt | CUES.get(anchor_key, set())

def _theme_chunks(theme):
    return sorted(fs.graph_store.get_chunks_for_theme(theme.theme_id))

def _abstract_frac(theme):
    cids = _theme_chunks(theme)
    if not cids:
        return 0.0
    nabs = sum(1 for c in fs.chunk_store.get_many(cids) if c.source_type == "abstract")
    return nabs / len(cids)

def _overlap(theme, cues):
    kw = {w.lower() for term in theme.model.keyword_terms for w in term.split()}
    return len(cues & kw)

def _sig(theme):
    return hashlib.sha1("".join(_theme_chunks(theme)).encode()).hexdigest()

def salient_theme(anchor_key):
    themes = anchors[anchor_key]["shelf"].themes()
    if not themes:
        return None
    cues = _cue_terms(anchor_key)
    pool = [t for t in themes if _overlap(t, cues) >= 1]
    if not pool:
        raise ValueError(
            f"{anchor_key}: no theme overlaps cues {sorted(cues)} — "
            "extend CUES or check the NB2 themes."
        )
    top = max(_overlap(t, cues) for t in pool)
    band = [t for t in pool if _overlap(t, cues) >= top - 1]   # near-top topical band
    prefer = PREFER.get(anchor_key, "abstract")
    def pref_score(t):
        af = _abstract_frac(t)
        return (1 - af) if prefer == "textbook" else af
    # deterministic full key: register, then overlap, then size, then chunk-set hash
    return max(band, key=lambda t: (round(pref_score(t), 3), _overlap(t, cues),
                                    t.model.chunk_count, _sig(t)))

chosen = {k: salient_theme(k) for k in anchors}
for k, th in chosen.items():
    cids = _theme_chunks(th) if th else []
    nabs = sum(1 for c in fs.chunk_store.get_many(cids) if c.source_type == "abstract")
    print(f"[{k}] chosen theme: {th.theme_id} {th.label if th else 'NONE'!r} "
          f"({th.model.chunk_count if th else 0} chunks, {nabs} abstracts, "
          f"prefer={PREFER.get(k)}) kw={list(th.model.keyword_terms)[:4] if th else []}")
assert all(chosen.values()), "an anchor has no theme — rerun NB2"

[olive_oil] chosen theme: olive_oil-leiden-fc736fe6 'healthy fats and oils' (27 chunks, 0 abstracts, prefer=abstract) kw=['oil', 'fats', 'fat', 'olive']
[legume] chosen theme: legume-leiden-54406f69 'legume based dietary intake' (43 chunks, 43 abstracts, prefer=abstract) kw=['dietary', 'intake', 'food', 'diet']
[fish] chosen theme: fish-leiden-247975f8 'fatty fish and pufas' (92 chunks, 92 abstracts, prefer=abstract) kw=['fish', 'dha', 'fish oil', 'pufas']
[dietary_fibre] chosen theme: dietary_fibre-leiden-d57e73e7 'dietary fiber intake' (75 chunks, 75 abstracts, prefer=abstract) kw=['intake', 'dietary', 'glucose', 'weight']


In [4]:
extracts = {}
cards = {}
for k, th in chosen.items():
    chunk_ids = list(fs.graph_store.get_chunks_for_theme(th.theme_id))
    texts = [c.text for c in fs.chunk_store.get_many(chunk_ids)]
    s1 = run_stage1(texts, summarizer,
                    map_reduce_threshold=cfg.map_reduce_threshold,
                    group_char_budget=cfg.group_char_budget)
    extracts[k] = {"theme_id": th.theme_id, "extract": s1.text,
                   "strategy": s1.strategy, "n_input_chunks": s1.n_input_chunks}
    adapter = _ThemeAdapter(theme_id=th.theme_id, label=th.label,
                            facet=th.model.facet, keyword_terms=list(th.model.keyword_terms))
    card = run_stage2(fs.llm, s1, adapter, chunk_ids, cfg)
    cards[k] = card
    print(f"[{k}] card built: {card.title!r} | evidence_quality={card.evidence_quality} | "
          f"cites {len(card.cited_chunk_ids)} chunks")
# persist the two cards into the graph/card stores
persist_cards(list(cards.values()), fs.graph_store, getattr(fs, "card_store", None))

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[olive_oil] card built: 'Healthy Fats and Oils' | evidence_quality=medium | cites 27 chunks


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[legume] card built: 'Legume Based Dietary Intake' | evidence_quality=high | cites 43 chunks


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[fish] card built: 'Fatty Fish and PUFAs' | evidence_quality=medium | cites 92 chunks


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[dietary_fibre] card built: 'Dietary Fiber Intake' | evidence_quality=medium | cites 75 chunks


## 3. Dump cards

In [5]:
for k, card in cards.items():
    rec = {
        "card_id": card.card_id, "target_id": card.target_id,
        "target_type": card.target_type, "title": card.title,
        "summary": card.summary, "tip": card.tip,
        "evidence_quality": card.evidence_quality,
        "controversy_note": card.controversy_note,
        "confidence_note": card.confidence_note,
        "evidence_sentences": list(card.evidence_sentences),
        "cited_chunk_ids": list(card.cited_chunk_ids),
        "llm_model": card.llm_model, "prompt_version": card.prompt_version,
        "safety_flagged": card.safety_flagged,
    }
    cs.save_json(OUT / f"{k}_card.json", rec)
    print(f"[{k}] {card.title}\n     {cs.excerpt(card.summary, 200)}\n")

[olive_oil] Healthy Fats and Oils
     Monounsaturated and polyunsaturated fats are considered 'good' fats as they lower harmful cholesterol levels and are beneficial for heart health. These fats are found in high amounts in olive oil, ca…

[legume] Legume Based Dietary Intake
     Research associates lower nutritional quality of diet with higher mortality risk. A higher intake of plant-based foods, including legumes, is linked to better health outcomes, such as higher folate s…

[fish] Fatty Fish and PUFAs
     The intake of fatty fish rich in n-3 polyunsaturated fatty acids (PUFAs), particularly eicosapentaenoic acid (EPA) and docosahexaenoic acid (DHA), has been associated with various health benefits. Th…

[dietary_fibre] Dietary Fiber Intake
     Research indicates that dietary fiber intake is associated with significant health benefits, including reduced glucose and insulin responses, improved weight management, and decreased body weight. A…



## 4. Provenance trace

In [6]:
for k, card in cards.items():
    cited = list(card.cited_chunk_ids)
    citations = []
    unresolved = []
    for cid in cited:
        c = fs.chunk_store.get(cid)
        if c is None:
            unresolved.append(cid)
            citations.append({"chunk_id": cid, "resolved": False, "source_doc": None, "excerpt": None})
        else:
            citations.append({"chunk_id": cid, "resolved": True,
                              "source_doc": c.source_doc_id, "excerpt": cs.excerpt(c.text, 240)})
    prov = {
        "anchor": k, "theme_id": card.target_id, "card_id": card.card_id,
        "cited_total": len(cited), "cited_resolved": len(cited) - len(unresolved),
        "citations": citations, "unresolved": unresolved,
    }
    cs.save_json(OUT / f"{k}_provenance.json", prov)
    pct = (prov["cited_resolved"] / prov["cited_total"] * 100) if prov["cited_total"] else 0.0
    print(f"[{k}] provenance: {prov['cited_resolved']}/{prov['cited_total']} resolved ({pct:.0f}%)"
          + (f"  UNRESOLVED: {unresolved}" if unresolved else ""))

[olive_oil] provenance: 27/27 resolved (100%)
[legume] provenance: 43/43 resolved (100%)
[fish] provenance: 92/92 resolved (100%)
[dietary_fibre] provenance: 75/75 resolved (100%)


## 5. M4 — extract vs card

In [7]:
for k in cards:
    card = cards[k]
    rec = {
        "anchor": k, "theme_id": card.target_id,
        "stage1_extract": extracts[k]["extract"],
        "stage1_strategy": extracts[k]["strategy"],
        "stage1_n_input_chunks": extracts[k]["n_input_chunks"],
        "card_title": card.title, "card_summary": card.summary, "card_tip": card.tip,
        "extract_chars": len(extracts[k]["extract"]),
        "card_summary_chars": len(card.summary),
    }
    cs.save_json(OUT / f"{k}_extract_vs_card.json", rec)
    print(f"[{k}] extract {rec['extract_chars']} chars ({rec['stage1_strategy']}) "
          f"-> card summary {rec['card_summary_chars']} chars")

[olive_oil] extract 813 chars (single) -> card summary 476 chars
[legume] extract 2193 chars (mapreduce) -> card summary 373 chars
[fish] extract 1544 chars (mapreduce) -> card summary 691 chars
[dietary_fibre] extract 1694 chars (mapreduce) -> card summary 436 chars


## 6. Card figures (title, summary, tip, evidence badge + cited chunks)

In [ ]:
import textwrap
from collections import Counter
# Evidence-quality enum is Literal["high","medium","low","debated","unclear"]
# (foodscholar.io.graph). The chip colour must key on THESE values.
BADGE = {"high": cs.COLORS["hierarchy"], "medium": cs.COLORS["amber"],
         "low": cs.COLORS["grey"], "debated": cs.COLORS["purple"],
         "unclear": cs.COLORS["grey"]}
SRC_COLOR = {"abstract": cs.COLORS["teal"], "textbook": cs.COLORS["amber"],
             "web": cs.COLORS["grey"]}

def source_type(cid, doc):
    """Source type for a cited chunk: prefer the chunk's own source_type, else
    infer from the doc id (doi/http -> abstract, *.pdf -> textbook)."""
    c = fs.chunk_store.get(cid)
    st = getattr(c, "source_type", None) if c is not None else None
    if st in ("abstract", "textbook", "web"):
        return st
    d = (doc or "").lower()
    if "doi.org" in d or d.startswith("http"):
        return "abstract"
    if d.endswith(".pdf"):
        return "textbook"
    return "web"

def card_fig(k):
    card = cards[k]
    prov = cs.load_json(OUT / f"{k}_provenance.json")
    resolved = [c for c in prov["citations"] if c["resolved"]]
    cited = resolved[:3]
    pct = (prov["cited_resolved"] / prov["cited_total"] * 100) if prov["cited_total"] else 0.0
    mix = Counter(source_type(c["chunk_id"], c["source_doc"]) for c in resolved)
    mix_str = " · ".join(f"{n} {t}{'s' if n != 1 else ''}" for t, n in mix.most_common())
    fig, ax = cs.slide(
        card.title,
        eyebrow=f"lens 4 · M4 · {k} card",
        caption=f"model: {card.llm_model} · provenance {prov['cited_resolved']}/{prov['cited_total']} resolved ({pct:.0f}%)",
    )
    eq = str(card.evidence_quality)
    cs.chip(ax, 0.84, 0.98, eq.upper(), color=BADGE.get(eq, cs.COLORS["grey"]), fontsize=12)
    y = 0.93
    # title on the card body + source-mix subline (both were missing before)
    ax.text(0.0, y, card.title, fontsize=16, weight="bold", color=cs.COLORS["ink"], va="top"); y -= 0.052
    ax.text(0.0, y, f"sources: {mix_str}", fontsize=11, color=cs.COLORS["muted"], va="top", style="italic"); y -= 0.055
    for line in textwrap.wrap(card.summary, 100)[:5]:
        ax.text(0.0, y, line, fontsize=12.5, va="top", color=cs.COLORS["ink"]); y -= 0.05
    if card.tip:
        y -= 0.015
        for line in textwrap.wrap("TIP — " + card.tip.strip(), 96)[:2]:
            ax.text(0.0, y, line, fontsize=12, va="top", color=cs.COLORS["teal"], style="italic"); y -= 0.05
    # confidence + controversy notes (were never drawn in the original figure)
    if card.controversy_note:
        y -= 0.012
        for line in textwrap.wrap("CONTROVERSY — " + card.controversy_note, 100)[:2]:
            ax.text(0.0, y, line, fontsize=11, va="top", color=cs.COLORS["purple"]); y -= 0.044
    if card.confidence_note:
        y -= 0.012
        for line in textwrap.wrap("CONFIDENCE — " + card.confidence_note, 100)[:3]:
            ax.text(0.0, y, line, fontsize=11, va="top", color=cs.COLORS["muted"]); y -= 0.044
    y -= 0.02
    ax.text(0.0, y, f"cited evidence ({prov['cited_resolved']}/{prov['cited_total']} resolved · {mix_str})",
            fontsize=12.5, weight="bold", color=cs.COLORS["muted"], va="top"); y -= 0.05
    for c in cited:
        st = source_type(c["chunk_id"], c["source_doc"])
        cs.chip(ax, 0.0, y - 0.012, st, color=SRC_COLOR[st], fontsize=9, weight="normal", pad=0.3)
        body = f"[{c['source_doc']}] {c['excerpt']}"
        for j, line in enumerate(textwrap.wrap(body, 98)[:2]):
            ax.text(0.085, y, line, fontsize=9.5, va="top", family="DejaVu Sans Mono", color=cs.COLORS["ink"]); y -= 0.040
        y -= 0.014
    cs.save_slide(fig, OUT / f"{k}_card.png")
    print("wrote", OUT / f"{k}_card.png")

for k in cards:
    card_fig(k)

## 7. summary.md

In [9]:
lines = ["# NB4 — Cards: faithfulness & provenance (M4) — summary", "",
         f"- LLM model: **{fs.llm.model_id}** (real; mock guard passed).", "", "## Cards"]
for k, card in cards.items():
    prov = cs.load_json(OUT / f"{k}_provenance.json")
    pct = (prov['cited_resolved'] / prov['cited_total'] * 100) if prov['cited_total'] else 0.0
    lines.append(f"- **{k}** — `{card.title}` (evidence: {card.evidence_quality}); "
                 f"provenance {prov['cited_resolved']}/{prov['cited_total']} resolved ({pct:.0f}%)"
                 + (f"; UNRESOLVED {prov['unresolved']}" if prov['unresolved'] else "") + ".")
lines += ["", "## Files", "env.json, {anchor}_card.json, {anchor}_provenance.json, "
          "{anchor}_extract_vs_card.json, {anchor}_card.png.", "",
          "## Deviations / limitations",
          "- Only the two salient anchor themes are sent to the LLM (two live calls), "
          "not the full facet — keeps the case study cheap/fast while exercising the "
          "real Stage-1→Stage-2 path.",
          "- Any unresolved citation or unsupported claim is listed above, not hidden.",
          "", "## Acceptance",
          "- [x] a card per anchor",
          "- [x] provenance resolution reported (deviations flagged)",
          "- [x] extract-vs-card available"]
(OUT / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))

# NB4 — Cards: faithfulness & provenance (M4) — summary

- LLM model: **llama-3.3-70b-versatile** (real; mock guard passed).

## Cards
- **olive_oil** — `Healthy Fats and Oils` (evidence: medium); provenance 27/27 resolved (100%).
- **legume** — `Legume Based Dietary Intake` (evidence: high); provenance 43/43 resolved (100%).
- **fish** — `Fatty Fish and PUFAs` (evidence: medium); provenance 92/92 resolved (100%).
- **dietary_fibre** — `Dietary Fiber Intake` (evidence: medium); provenance 75/75 resolved (100%).

## Files
env.json, {anchor}_card.json, {anchor}_provenance.json, {anchor}_extract_vs_card.json, {anchor}_card.png.

## Deviations / limitations
- Only the two salient anchor themes are sent to the LLM (two live calls), not the full facet — keeps the case study cheap/fast while exercising the real Stage-1→Stage-2 path.
- Any unresolved citation or unsupported claim is listed above, not hidden.

## Acceptance
- [x] a card per anchor
- [x] provenance resolution reported (deviation